# **P4: Project Synthesis**



---
**DSA 405 · Fall 2026 · Project Milestone 4**


# DSA 405 – Project Milestone 4: Project Synthesis

**Project:** Animal Shelter Placement Times  
**Repository:** https://github.com/SoumyaPannala/dsa405-project

## 1. Data Motivation

This project investigates the question: **What animal characteristics are associated with faster placement into a home?**

I chose this topic because animal shelters handle large numbers of animals, and understanding what is associated with shorter placement times could help shelters improve their adoption and placement efforts.

My original question was phrased slightly differently to focus specifically on adoption. However, after examining the available data, I decided to include three outcomes: Adoption, Rto-Adopt, and Return to Owner. I broadened the question to placement into a home because all three outcomes represent an animal leaving the shelter to live with someone, although they involve different processes.

I focused on characteristics such as age, animal type, breed, and color, while also examining intake condition and differences between outcome types.

## 2. Sources, Provenance, & Terms

I used two publicly available datasets from the City of Austin Open Data Portal.

### Source 1: Austin Animal Center Intakes

This dataset was produced and published by the City of Austin through Austin Animal Services as part of its public open-data program. It documents animals entering the shelter and supports the department's work caring for lost and homeless animals, tracking shelter activity, measuring performance, and providing information to the public.

The dataset contains intake records from October 1, 2013 through May 4, 2025. Each record includes information such as animal ID, name, intake date and location, intake type, condition, animal type, sex, age, breed, and color. It contains 173,812 records, and animals may appear more than once because they can have multiple intake events.

I accessed the dataset through the portal's documented API and downloaded it on September 25, 2026.

Source URL: https://data.austintexas.gov/api/v3/views/wter-evkm/query.json?accessType=DOWNLOAD

### Source 2: Austin Animal Center Outcomes

This dataset was also produced by the City of Austin through Austin Animal Services and published through the City of Austin Open Data Portal. It documents animal outcomes, including the dates and types of outcomes recorded by the shelter.

The dataset contains 173,775 outcome records. I accessed and downloaded the dataset through the portal's public CSV export endpoint on November 20, 2026.

Source URL: https://data.austintexas.gov/api/v3/views/9t4d-g238/export.csv

### Ethics and Legality in Practice

Both datasets are listed as Public Domain. The City of Austin Open Data Terms of Use state that data available through the portal are offered free of charge and without restriction, while requesting attribution to both the City of Austin and the city department responsible for the data. I therefore credit the City of Austin and Austin Animal Services.

The terms also prohibit attempts to re-identify individuals represented in the data, contact those individuals, or combine datasets for the purpose of re-identification. My project focuses on shelter animals and their recorded characteristics and outcomes, rather than identifying or contacting individuals. I do not attempt to re-identify anyone represented in the records.

I also reviewed the portal's `robots.txt` file, which specifies a one-second crawl delay for all user agents. I followed this guidance by including a one-second delay before making the download request. My code also uses a User-Agent identifying the request as part of a DSA 405 student project. The acquisition used the portal's public CSV export endpoint and did not require authentication or circumvent any access restrictions.

There was one change from my original A8 plan. I initially planned to use the dataset's API query endpoint, which worked during the A8 feasibility test. However, when I attempted to use it again for P3, it returned a 403 authentication error. The error indicated that authentication or an application token was required. Rather than attempting to bypass that requirement, I switched to the portal's public CSV export endpoint to access that second dataset. This allowed me to download the same Animal Outcomes dataset without authentication. The source dataset itself did not change; only the acquisition method changed.

The CSV export returned the complete dataset in one response, so pagination was not necessary. I also checked the response headers for rate-limit information, but the server did not return any explicit rate-limit headers. I therefore followed the one-second crawl delay specified in `robots.txt`.

## 3. Data Dictionary & Metadata

The project uses intake and outcome records that were combined into a single dataset. The final analysis dataset, `animal_placements_valid`, contains 110,673 records and 28 variables. It includes animal identifiers, intake information, outcome information, and calculated placement times.

The data dictionary below documents the variables in the final analysis dataset, including their data types, units, categorical values, valid ranges, missingness, and relevant notes.

| Variable name | Type | Units | Factor levels | Valid range | Missingness | Notes |
|---|---|---|---|---|---|---|
| `:id` | String | None | N/A | Unique source-row identifiers | 0 (0%) | Unique identifier for the original intake record. Retained during cleaning because removing it caused duplication problems. |
| `animal_id` | String | None | N/A | Animal IDs beginning with A followed by digits | 0 (0%) | Identifies an animal. Not unique because an animal may have multiple intake and outcome records. |
| `name` | String | None | N/A | Recorded animal names | 14,547 (13.14%) | Name recorded in the intake dataset. Missing values indicate that no name was recorded. |
| `datetime` | Datetime | Date and time (UTC) | N/A | Intake timestamps | 0 (0%) | Date and time the animal entered the shelter. |
| `found_location` | String | None | N/A | Recorded locations | 0 (0%) | Location where the animal was found or picked up. |
| `intake_type` | Categorical | None | Stray; Owner Surrender; Public Assist; Wildlife; Abandoned; Euthanasia Request | Six recorded categories | 0 (0%) | Describes the circumstances associated with the animal's intake. |
| `intake_condition` | Categorical | None | Normal; Injured; Sick; Nursing; Neonatal; Medical; Aged; Other; Pregnant; Feral; Behavior; Med Attn; Unknown; Med Urgent; Neurologic; Panleuk | 16 recorded categories | 0 (0%) | Describes the animal's recorded condition upon intake. |
| `animal_type` | Categorical | None | Dog; Cat; Other; Bird; Livestock | Five recorded categories | 0 (0%) | Animal type recorded in the intake dataset. |
| `sex_upon_intake` | Categorical | None | Intact Male; Intact Female; Neutered Male; Spayed Female; Unknown | Five recorded categories | 1 (0.00%) | Sex and reproductive status recorded at intake. Unknown is a valid category, not a missing value. |
| `age_upon_intake` | String | Years, months, weeks, or days | N/A | 46 distinct recorded age values | 7 (0.01%) | Animal's age at intake, recorded as a number and unit. Used to derive `age_years`. |
| `breed` | String | None | N/A | 2,593 distinct recorded values | 0 (0%) | Breed recorded in the intake dataset. |
| `color` | String | None | N/A | 574 distinct recorded values | 0 (0%) | Animal's color recorded in the intake dataset. |
| `age_years` | Float | Years | N/A | 0–28 years | 7 (0.01%) | Derived from `age_upon_intake` by converting the recorded age into numerical years. |
| `event_number` | Integer | None | N/A | 1–33 | 0 (0%) | Chronological event number assigned within each animal ID. Used with `animal_id` to match intake and outcome records. |
| `Date of Birth` | String | Date | N/A | Recorded dates of birth | 0 (0%) | Date of birth recorded in the outcome dataset. |
| `Name` | String | None | N/A | Recorded animal names | 14,547 (13.14%) | Name recorded in the outcome dataset. Corresponds to the intake-side `name` field. |
| `DateTime` | Datetime | Date and time (UTC) | N/A | Outcome timestamps | 0 (0%) | Date and time the animal's outcome was recorded. |
| `MonthYear` | String | Month and year | N/A | 140 distinct recorded month-year values | 0 (0%) | Month and year associated with the outcome record, formatted as MM-YYYY. |
| `Outcome Type` | Categorical | None | Adoption; Return to Owner; Rto-Adopt | Three recorded categories in final dataset | 0 (0%) | Outcome category. These three categories were retained because they represent placement-related outcomes within the scope of the research question. |
| `Outcome Subtype` | Categorical | None | Foster; Offsite; Prc; Field; Barn; At Vet; Customer S | Seven recorded categories | 92,177 (83.28%) | Additional classification associated with an outcome. Missingness indicates that no subtype was recorded. |
| `Animal Type` | Categorical | None | Dog; Cat; Other; Bird; Livestock | Five recorded categories | 0 (0%) | Animal type recorded in the outcome dataset. |
| `Sex upon Outcome` | Categorical | None | Intact Male; Intact Female; Neutered Male; Spayed Female; Unknown | Five recorded categories | 1 (0.00%) | Sex and reproductive status recorded at outcome. |
| `Age upon Outcome` | String | Years, months, weeks, or days | N/A | 50 distinct recorded age values | 0 (0%) | Animal's age at the time of the outcome, recorded as a number and unit. |
| `Breed` | String | None | N/A | 2,593 distinct recorded values | 0 (0%) | Breed recorded in the outcome dataset. |
| `Color` | String | None | N/A | 574 distinct recorded values | 0 (0%) | Animal's color recorded in the outcome dataset. |
| `exact_days_to_outcome` | Float | Days | N/A | 0–1,912.94 days | 0 (0%) | Exact elapsed time between intake and outcome, calculated from the two timestamps. Used as the primary measure of placement time. |
| `days_to_outcome` | Integer | Days | N/A | 0–1,913 days | 0 (0%) | Elapsed time between intake and outcome rounded to whole days. Retained alongside the more precise measure. |
| `age_group` | Categorical | Years | Under 1; 1–under 3; 3–under 8; 8+ | Four ordered categories | 7 (0.01%) | Derived from `age_years` to group animals by age for comparisons. Missing values correspond to records without a recorded intake age. |

**Variables removed during cleaning**

The following four variables were present in the original intake dataset but were not retained in the final analysis dataset.

| Variable name | Type | Units | Factor levels | Valid range | Missingness | Notes |
|---|---|---|---|---|---|---|
| `:version` | Integer | None | N/A | Source-system version values | Not applicable | System metadata; removed because it was not relevant to the analysis. |
| `:created_at` | Datetime | Date and time | N/A | Source-record creation timestamps | Not applicable | System metadata; removed because it was not relevant to the analysis. |
| `:updated_at` | Datetime | Date and time | N/A | Source-record update timestamps | Not applicable | System metadata; removed because it was not relevant to the analysis. |
| `datetime2` | Datetime | Date and time | N/A | Same values as `datetime` | Not applicable | Removed because it contained the same timestamp values as `datetime` in every row, making it redundant. |

## 4. Data Cleaning Steps Narrative

I began by loading the raw Austin Animal Center Intakes and Outcomes datasets. The intake dataset contained 173,812 records and 16 columns, while the outcome dataset contained 173,775 records and 16 columns. I first examined the structure of both datasets, including their column names, data types, missing values, and record counts, to understand the information available and identify potential data-quality issues.

### Intake Data Cleaning

I inspected the intake dataset for redundant columns, missing values, and inconsistent entries. I found that the datetime2 column contained the same information as the existing datetime column, so I removed the redundant column. I also removed three source-system metadata columns: :version, :created_at, and :updated_at, because they were not needed for my analysis. I initially tried removing :id as well, but this caused duplication problems, so I retained it to preserve the unique source identifier for each record. The cleaned dataset therefore contained 13 columns, including the newly derived age_years column.

I then examined the age information recorded in `age_upon_intake`. Some entries contained negative age values or inconsistent formats, such as `"1 weeks"`. I addressed these issues and standardized the age values before converting them into a numerical age measurement in years. I stored this calculated value in a new `age_years` column. I used verification checks to confirm that negative ages and the identified inconsistent entry were no longer present.

### Combining Intake and Outcome Records

Next, I prepared the intake and outcome timestamps so that placement times could be calculated. Because individual animals could have multiple intake and outcome records, matching records using only `animal_id` could result in incorrect pairings.

To address this, I assigned a chronological event number to each record within each animal ID group using `groupby()` and `cumcount()`. I then combined the datasets using both `animal_id` and the event number. This created 172,850 matched intake and outcome pairs.

### Calculating and Validating Placement Times

After combining the datasets, I calculated the exact time difference between each intake and its corresponding outcome using the recorded timestamps. I retained this precise interval in the `exact_days_to_outcome` column and also maintained a rounded days-to-outcome value for other uses.

I checked the calculated intervals for invalid values. In the final placement subset, I identified 79 records with negative time intervals, meaning the outcome timestamp occurred before the intake timestamp. I excluded these records from the placement analysis because they did not represent valid placement times. However, I retained zero-day intervals because an animal could enter and leave the shelter on the same day.

### Preparing the Final Analysis Dataset

I then filtered the combined records to focus on three placement-related outcomes: Adoption, Rto-Adopt, and Return to Owner. I excluded other outcomes, including Transfer, Euthanasia, Died, Disposal, Missing, Relocate, Stolen, and Lost, because they were outside the scope of our research question. I excluded these outcomes because they did not represent an animal being placed into a home through one of the three outcome types included in my research question. Including them would have expanded the scope of the project to other shelter outcomes that I could not meaningfully examine without additional information and data.

After filtering the outcomes and removing the invalid negative intervals, I obtained a final dataset of **110,673 valid placement records**. I also found seven records with missing age information. These records remained available for analyses that did not require age, but were excluded from the age-group comparison.

Finally, I created an `age_group` variable to compare placement times across four age categories: Under 1, 1–under 3, 3–under 8, and 8+. I used the cleaned age values to create these groups and then examined placement times across age, animal type, breed, color, intake condition, and outcome type.

Throughout the process, I used checks and assertions to verify record counts, identify invalid values, and confirm that the cleaned data met expectations. This helped ensure that the final analysis was based on consistent and valid placement records.

*This section has described the path from the raw data to the final dataset, along with the decisions I made along the way. For the full, updated cleaning log and corresponding code, please see the Cleaning Log section of the P3 notebook, which has since been updated.*

## 5. Integration & Verification

I combined the intake and outcome datasets using the animal ID and a chronological event number. The event number was created separately for each animal to help match repeated intake and outcome records in order.

The original intake dataset contained 173,812 records, while the outcome dataset contained 173,775 records. The matching process produced 172,850 matched pairs.

I verified the integration by checking the number of records, confirming that each animal ID and event number combination was unique, checking for missing intake and outcome timestamps, and verifying that the calculated time intervals were valid.

I also tested the verification code by deliberately adding a duplicate record to a test copy of the dataset. The assertion correctly detected the unexpected record count, confirming that the check could identify at least one type of data problem.

One limitation is that matching records chronologically does not guarantee that every intake was paired with its exact corresponding outcome. The animal ID identifies an animal, but it does not necessarily identify a specific shelter episode. To reduce this risk, I assigned each intake and outcome record a chronological event number within its animal ID group and matched records using both the animal ID and event number. This is more reliable than matching on animal ID alone because it helps keep repeated shelter visits in the correct order. However, it still relies on the assumption that intake and outcome records are complete and follow the same chronological sequence. If a record is missing, delayed, or out of order, an intake could still be paired with the wrong outcome. Therefore, this method provides a consistent approach to matching records, but I cannot guarantee that every matched pair represents the exact same shelter episode.

## 6. Findings

The analysis showed several patterns in placement times.

- **Age:** Older animals had shorter median placement times. Animals under one year old had a median of about 11.3 days, compared with about 4.2 days for animals eight years and older. Although median placement times generally decreased with age, the mean placement times did not follow the same pattern. This suggests that some older animals experienced particularly long stays, which increased the average despite their shorter median placement time.
- **Animal type:** Dogs had a median placement time of about 6 days, compared with about 21 days for cats. Birds and other animals had median placement times of about 6.9 and 11.9 days, respectively. Livestock had a longer median of about 35.8 days, although this category contained only 23 records.
- **Breed:** Among common dog breeds, median placement times ranged from approximately 5 to 13 days. Among common cat breeds, they ranged from approximately 13 to 24 days.
- **Color:** Median placement times varied by color, although the differences were generally smaller for dogs than for cats.
- **Intake condition:** Animals recorded as Normal had a median placement time of about 8 days, compared with approximately 22 days for Injured animals and 23 days for Sick animals. Nursing and Neonatal animals had especially long median placement times of about 58 and 62 days, respectively. Aged animals were an exception, with a median of about 2 days.
- **Outcome type:** Return to Owner cases had the shortest median time at about 1.7 days, followed by Rto-Adopt at about 7.9 days and Adoption at about 14.1 days.

I also compared dogs and cats within each outcome type. Dogs had shorter median placement times in all three categories. For example, the median for dog adoptions was about 9.3 days, compared with about 25.2 days for cat adoptions.

Overall, the results suggest that animal type, age, breed, color, intake condition, and outcome type are associated with placement time in this dataset.

There are several possible explanations for these patterns. Dogs may receive different levels of adoption interest than cats, while older animals may already have some training or be easier to care for. Animals entering the shelter in poorer condition may need additional care before placement, while nursing and neonatal animals may require more specialized attention. Return to Owner cases are likely faster because an owner may already be looking for the animal. Breed and color differences could also reflect adopter preferences or differences in the animals entering the shelter. However, these explanations are possibilities rather than conclusions directly supported by the available data.

## 7. What Is Missing?

There are several limitations to consider when interpreting these results.

First, the analysis only includes animals with matched intake and outcome records. Animals without a successful match between the two datasets are not represented in the final analysis.

Second, my placement analysis only includes Adoption, Rto-Adopt, and Return to Owner. Other outcomes, such as transfers, euthanasia, and death, were not included. This means our results do not describe the experiences of every animal entering the shelter.

Third, the chronological matching process cannot guarantee that every intake record was paired with its exact corresponding outcome, particularly for animals with multiple shelter visits.

I also cannot determine why an animal was placed faster or slower. The datasets do not provide enough information about adopter interest, animal behavior, training, marketing exposure, or the circumstances of an owner's search. So while I can identify patterns among animals with shorter placement times, the underlying reasons for those patterns would be speculative.

Additionally, breed and color comparisons focused on dogs and cats because they represented the overwhelming majority of placement records. Other animal types had much smaller sample sizes. Some intake-condition categories also had very few records, so I limited my comparison to categories with at least 300 records.

One question this dataset cannot answer is: **Would increasing adoption visibility or changing shelter outreach strategies actually reduce placement times for animals that tend to stay longer?** Answering this question would require additional information about adoption campaigns, adopter behavior, and shelter practices. We would first need to examine which outreach strategies tend to be most effective and under what circumstances. That could then help inform a future study investigating whether implementing those strategies actually reduces placement times.

## 8. AI Use Log & Reflection

I used ChatGPT as an AI assistant throughout the project. It helped with understanding Python and pandas operations, developing and troubleshooting data cleaning and verification code, interpreting analysis results, creating visualizations, and organizing the final report.

One example of an AI mistake occurred during visualization development. ChatGPT initially suggested using a breed column name that did not match the actual DataFrame, resulting in a KeyError. I checked the actual column names and corrected the code using the lowercase intake-side `breed` column.

Another issue occurred when an initial visualization only included dog breeds, even though I wanted to compare both dogs and cats. I noticed the omission when reviewing the breeds shown in the chart and scanning the code to see if it included only dogs. Finding confirmation, I revised the chart to include cats as well.

These experiences showed that AI suggestions can be useful, but they still need to be checked against the actual data and project goals. Especially since some problems are harder to detect than these two mistakes I mentioned here, it's essential to ensure that AI-generated code is not only correct and applicable for the data, but that it is consistent with your intentions. Often, AI can cause silent issues like fan-out joins and other errors. While I didn't experience that one, I was aware that it could happen and made sure to thoroughly run code with assertions to check that results and major joins and other steps matched with what I expected to get. I used the notebook outputs and my understanding of the dataset to identify and correct these problems rather than assuming every suggestion ChatGPT gave was accurate.

## 9. Reproducibility

The complete project repository is available at:

https://github.com/SoumyaPannala/dsa405-project

The repository contains the raw data, analysis notebooks, and supporting documentation needed to reproduce the project.

To reproduce the analysis:

1. Clone or download the repository.
2. Open the notebook in Jupyter Notebook, JupyterLab, or Google Colab.
3. Run the notebook from beginning to end, starting with the raw data and continuing through cleaning, integration, analysis, and visualizations. Start with P1, and continue in order to P2, and P3. P4 is this accompanying written report and does not contain any executable code sections.
- P1 does the initial work to create a research question, find data, and ensure that it is suitable for this project.
- P2 audits the intakes data and cleans it in preparation for the join in P3.
- P3 downloads the outcomes data and joins it to the intakes dataset. It also now has an additional section that completes the analysis and includes charts to answer the research question. P3 also includes a section after the analysis with extra documentation, where you may find the cleaning log (updated from P2) and corresponding code.
- P4 is this written report you are reading currently.

The notebooks were all tested from a restarted kernel and ran successfully without errors.